# Extração de Características linguísticas

In [1]:
import os
import shutil
import gc
from pathlib import Path
from time import perf_counter
import math

import pandas as pd
import numpy as np
import spacy
import lftk
from tqdm import tqdm
from sklearn.tree import DecisionTreeClassifier

In [2]:
BASE_DIR = Path.cwd().parent 

FILE_PATH = BASE_DIR / "data/preprocessed_shorts.csv" 
OUTPUT_PATH = BASE_DIR / "data/lftk_output_features.parquet"
OUTPUT_PATH_FILTERED = BASE_DIR / "data/lftk_filtered_features.parquet"
TEMP_DIR = BASE_DIR / "data/temp_partitions"

COLUMN = "transcription"
TARGET_COLUMN = "topico" 
BATCH_SIZE = 32   
CHUNK_SIZE = 1000

In [3]:
def stream_lftk_features(texts: list, nlp: spacy.Language, features: list) -> pd.DataFrame:
    """Extrai features em formato de fluxo (stream) para economizar memória RAM."""
    results = []
    doc_generator = nlp.pipe(texts, batch_size=BATCH_SIZE, disable=["ner"])
    
    for doc in tqdm(doc_generator, total=len(texts), desc="spaCy + LFTK"):
        extractor = lftk.Extractor(docs=doc)
        extractor.customize(stop_words=True, punctuations=True)   
        feats = extractor.extract(features=features)
        results.append(feats)
        
    return pd.DataFrame(results)

In [4]:
def calculate_gini_gain(df: pd.DataFrame, feature_cols: list, target_col: str) -> pd.Series:
    """
    Calcula o Gini Gain usando árvores de decisão (stumps) para encontrar 
    o melhor limiar de separação de forma otimizada.
    """
    print("\nCalculando Gini Gain para todas as features...")
    gains = {}
    
    # Tratamento de NaNs
    df_clean = df.dropna(subset=feature_cols + [target_col])
    y = df_clean[target_col]
    
    for col in tqdm(feature_cols, desc="Gini Gain"):
        X = df_clean[[col]]
        # encontra o limiar que maximiza o Gini Gain
        tree = DecisionTreeClassifier(max_depth=1, criterion='gini', random_state=42)
        tree.fit(X, y)
        
        # O decréscimo de impureza total da árvore é equivalente ao Gini Gain absoluto
        gini_gain = tree.tree_.compute_feature_importances(normalize=False)[0]
        gains[col] = gini_gain
        
    return pd.Series(gains).sort_values(ascending=False)

In [5]:
def filter_features_by_spearman(df: pd.DataFrame, ranked_features: pd.Series, threshold: float = 0.7) -> list:
    """
    Filtra features redundantes usando Correlação de Spearman, 
    mantendo as de maior Gini Gain.
    """
    print(f"\nFiltrando features com Correlação de Spearman > {threshold}...")
    selected_features = []
    features_to_eval = list(ranked_features.index)
    
    # Matriz de correlação de Spearman
    corr_matrix = df[features_to_eval].corr(method='spearman').abs()
    
    with tqdm(total=len(features_to_eval), desc="Filtragem Spearman") as pbar:
        while features_to_eval:
            # Pega a feature com maior Gini Gain que ainda está na lista
            top_feat = features_to_eval.pop(0)
            selected_features.append(top_feat)
            
            # Identifica as features altamente correlacionadas a ela
            correlated = corr_matrix.index[(corr_matrix[top_feat] > threshold)].tolist()
            
            # Remove as correlacionadas da fila de avaliação
            initial_len = len(features_to_eval)
            features_to_eval = [f for f in features_to_eval if f not in correlated and f != top_feat]
            pbar.update(initial_len - len(features_to_eval) + 1)
            
    print(f"-> Features reduzidas de {len(ranked_features)} para {len(selected_features)}.")
    return selected_features

In [6]:
df = pd.read_csv(FILE_PATH)
nlp = spacy.load("pt_core_news_lg")

all_features = lftk.search_features(return_format="list_key")
print(f"Total de features configuradas: {len(all_features)}")

# Processamento LFTK em Lotes
TEMP_DIR.mkdir(parents=True, exist_ok=True)
temp_files = []
total_chunks = math.ceil(len(df) / CHUNK_SIZE)

Total de features configuradas: 220


### Extração LFTK em lotes

In [15]:
for i in range(total_chunks):
        print(f"\nLote {i+1}/{total_chunks}")
        start_idx = i * CHUNK_SIZE
        chunk_df = df.iloc[start_idx : start_idx + CHUNK_SIZE].reset_index(drop=True)
        chunk_texts = chunk_df[COLUMN].astype(str).tolist()

        start_time = perf_counter()
        df_feats = stream_lftk_features(chunk_texts, nlp, all_features)
        print(f"Lote finalizado em {perf_counter() - start_time:.2f}s")

        chunk_final = pd.concat([chunk_df, df_feats], axis=1)
        temp_file = TEMP_DIR / f"part_{i:04d}.parquet"
        chunk_final.to_parquet(temp_file, engine='fastparquet')
        temp_files.append(temp_file)

        del chunk_df, chunk_texts, df_feats, chunk_final
        gc.collect()

df_final = pd.concat([pd.read_parquet(f, engine='fastparquet') for f in temp_files], ignore_index=True)

gc.collect()
shutil.rmtree(TEMP_DIR, ignore_errors=True)


Lote 1/10


spaCy + LFTK: 100%|██████████| 1000/1000 [15:01<00:00,  1.11it/s]  


Lote finalizado em 901.91s

Lote 2/10


spaCy + LFTK: 100%|██████████| 1000/1000 [03:36<00:00,  4.62it/s]


Lote finalizado em 216.66s

Lote 3/10


...

Lote finalizado em 228.11s

Lote 9/10


spaCy + LFTK: 100%|██████████| 1000/1000 [03:49<00:00,  4.35it/s]


Lote finalizado em 229.89s

Lote 10/10


spaCy + LFTK: 100%|██████████| 270/270 [01:04<00:00,  4.19it/s]


Lote finalizado em 64.40s


In [16]:
df_final.to_parquet(OUTPUT_PATH, engine='fastparquet')

In [ ]:
df_final = pd.read_parquet(OUTPUT_PATH, engine='fastparquet')

print("\nGini Gain")
feature_cols = [col for col in all_features if col in df_final.columns]
ranked_features = calculate_gini_gain(df_final, feature_cols, TARGET_COLUMN)
print("\nTop 10 features pelo Gini Gain:")
print(ranked_features.head(10))

final_selected_features = filter_features_by_spearman(df_final, ranked_features, threshold=0.7)

cols_to_keep = list(df.columns) + final_selected_features
df_filtered = df_final[cols_to_keep]

df_filtered.to_parquet(OUTPUT_PATH_FILTERED, engine='fastparquet')

print(f"\nConcluído com sucesso!")
print(f"Foram selecionadas as top {len(final_selected_features)} características linguísticas.")
print(f"Dataset filtrado salvo em: {OUTPUT_PATH_FILTERED}")


Gini Gain

Calculando Gini Gain para todas as features...


KeyError: ['topico']